# Can we predict 30-day readmission (`readmit_30d`)?

Plan: inspect the data, then compare a baseline against logistic regression and gradient boosting with **person-grouped** cross-validation (the same person appears in several rows, so a random split would leak). Metrics: ROC-AUC and PR-AUC (classes are imbalanced).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv')
pd.read_csv('columns.csv')

In [ ]:
print(df.shape, 'persons:', df.person_key.nunique())
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print(df.readmit_30d.value_counts(normalize=True))
print(df.groupby('person_key').size().describe())
df.describe().T

## Data checks
- `person_key` is an identifier: used only for grouping, **not** as a feature.
- Rows from one person are correlated, so all CV is grouped by person.
- `postcode_area` is a numeric code of an area, so it is treated as categorical where used.

In [ ]:
num = ['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
print(df.groupby('readmit_30d')[num].mean().T)
fig, ax = plt.subplots(2, 4, figsize=(16, 7)); ax = ax.ravel()
for a, c in zip(ax, num):
    q = pd.qcut(df[c], 10, duplicates='drop'); r = df.groupby(q, observed=True).readmit_30d.mean()
    a.plot(range(len(r)), r.values, 'o-'); a.set_title(c); a.set_ylim(0, None); a.axhline(df.readmit_30d.mean(), ls='--', c='gray')
r = df.groupby('sex').readmit_30d.mean(); ax[7].bar(r.index, r.values); ax[7].set_title('sex')
fig.suptitle('Readmission rate by decile (dashed = overall)'); plt.tight_layout(); plt.show()

In [ ]:
g = df.groupby('postcode_area').readmit_30d.agg(['mean', 'size']); print(g['size'].describe())
print(g.sort_values('mean').iloc[[0, 1, 2, -3, -2, -1]])

## Models (grouped 5-fold CV, out-of-fold predictions)

In [ ]:
y = df.readmit_30d.values; groups = df.person_key.values
feats = num + ['sex', 'postcode_area']; X = df[feats]
cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), ['sex'])])
pre_pc = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), ['sex', 'postcode_area'])])
models = {
    'baseline (prevalence)': DummyClassifier(strategy='prior'),
    'logreg (no postcode)': make_pipeline(pre, LogisticRegression(max_iter=2000)),
    'logreg (+postcode)': make_pipeline(pre_pc, LogisticRegression(max_iter=2000, C=0.3)),
    'hist-GB': make_pipeline(ColumnTransformer([('p', 'passthrough', num), ('s', OneHotEncoder(), ['sex'])]),
                             HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED)),
}
oof = {}; rows = []
for k, m in models.items():
    p = cross_val_predict(m, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]; oof[k] = p
    fa = [roc_auc_score(y[t], p[t]) for _, t in cv.split(X, y, groups)]
    rows.append(dict(model=k, roc_auc=roc_auc_score(y, p), auc_fold_sd=np.std(fa), pr_auc=average_precision_score(y, p), brier=brier_score_loss(y, p)))
res = pd.DataFrame(rows).set_index('model').round(3); print('prevalence =', y.mean().round(3)); res

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, p in oof.items():
    if 'baseline' in k: continue
    f, t, _ = roc_curve(y, p); ax[0].plot(f, t, label=k)
    pr, rc, _ = precision_recall_curve(y, p); ax[1].plot(rc, pr, label=k)
    fp, mp = calibration_curve(y, p, n_bins=10, strategy='quantile'); ax[2].plot(mp, fp, 'o-', label=k)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(y.mean(), c='k', ls=':'); ax[1].set(title='Precision-Recall', xlabel='Recall', ylabel='Precision')
ax[2].plot([0, 1], [0, 1], 'k:'); ax[2].set(title='Calibration', xlabel='Predicted', ylabel='Observed')
ax[0].legend(); plt.tight_layout(); plt.show()

## What drives the prediction? (permutation importance on held-out persons)

In [ ]:
tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=SEED).split(X, y, groups))
m = models['hist-GB'].fit(X.iloc[tr], y[tr]); print('holdout AUC', round(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]), 3))
pi = permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, feats).sort_values(); err = pd.Series(pi.importances_std, feats)[imp.index]
imp.plot.barh(xerr=err); plt.xlabel('drop in AUC when shuffled'); plt.show()
lr = models['logreg (no postcode)'].fit(X, y)
print(pd.Series(lr[-1].coef_[0], lr[0].get_feature_names_out()).round(3).sort_values())

In [ ]:
p = oof['hist-GB']; thr = np.quantile(p, 0.8); flag = p >= thr
print(f'Top 20% flagged: readmit rate {y[flag].mean():.3f} vs {y[~flag].mean():.3f} for the rest (overall {y.mean():.3f});',
      f'captures {y[flag].sum()/y.sum():.0%} of readmissions')

## Does a person's history help?
Rows are repeat stays and `postcode_area` is constant within a person. Person-level outcome variance is much larger than chance (below), so people differ in risk. Earlier stays are known when a later stay ends, so we can build history features **using only earlier visits** (`visit_number` is contiguous from 1 for everybody).

Caveat: this assumes visit order follows time and that the earlier label is known at prediction time. There are no dates in the data, so that cannot be checked here.

In [ ]:
pm = df.groupby('person_key').readmit_30d.agg(['mean', 'size'])
print('variance of person mean readmit rate:', round(pm['mean'].var(), 4), '| expected if all persons had equal risk:', round((y.mean()*(1-y.mean())/pm['size']).mean(), 4))
h = df.sort_values(['person_key', 'visit_number']).copy()
gg = h.groupby('person_key').readmit_30d
h['prior_n'] = gg.cumcount()
h['prior_readmits'] = gg.cumsum() - h.readmit_30d          # strictly earlier visits only
h['prior_rate'] = h.prior_readmits / h.prior_n.replace(0, np.nan)
h['prev_readmit'] = gg.shift(1)
h = h.loc[df.index]                                       # back to original row order
hist = ['prior_n', 'prior_readmits', 'prior_rate', 'prev_readmit']
print(h.groupby('prior_readmits').readmit_30d.agg(['mean', 'size']).head(7))

In [ ]:
cur = num + ['sex']
mk = lambda cols: make_pipeline(ColumnTransformer([('p', 'passthrough', [c for c in cols if c != 'sex']), ('s', OneHotEncoder(), ['sex'])]),
        HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=SEED))
rows = []; oof2 = {}
for name, cols in [('current stay only', cur), ('current stay + history', cur + hist)]:
    p = cross_val_predict(mk(cols), h[cols], y, cv=cv, groups=groups, method='predict_proba')[:, 1]; oof2[name] = p
    fa = [roc_auc_score(y[t], p[t]) for _, t in cv.split(X, y, groups)]
    rows.append(dict(model=name, roc_auc=roc_auc_score(y, p), auc_fold_sd=np.std(fa), pr_auc=average_precision_score(y, p), brier=brier_score_loss(y, p)))
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
p = oof2['current stay + history']; flag = p >= np.quantile(p, 0.8)
print(f'Top 20% flagged: readmit rate {y[flag].mean():.3f} vs {y[~flag].mean():.3f} for the rest; captures {y[flag].sum()/y.sum():.0%} of readmissions')
fp, mp = calibration_curve(y, p, n_bins=10, strategy='quantile')
plt.plot(mp, fp, 'o-'); plt.plot([0, 1], [0, 1], 'k:'); plt.xlabel('Predicted'); plt.ylabel('Observed'); plt.title('Calibration, with history'); plt.show()

## Conclusions
- **From the current stay alone, prediction is weak:** grouped-CV ROC-AUC ~0.63-0.65 (chance = 0.50), PR-AUC ~0.32 vs 0.20 prevalence. Length of stay, lab A and number of medications carry the signal (higher -> more readmission); age, BMI, lab B, sex and visit number add essentially nothing. Logistic regression is as good as or better than gradient boosting, so the signal is roughly linear and small.
- **`postcode_area` looks significant in a naive test but does not help out-of-sample.** The naive chi-square ignores that rows cluster within persons (postcode is fixed per person), and adding it lowered CV performance.
- **Prior-visit outcomes are a strong predictor:** adding history (earlier readmissions) raises ROC-AUC to ~0.82 and the top-20% risk group captures a much larger share of readmissions. This holds only if earlier labels are available at prediction time and visits are in time order, which should be confirmed with the data owner (no dates here).
- Always evaluate grouped by person; a random row split would leak person-level information and look better than reality.
- Not done: threshold choice (depends on intervention cost), hyperparameter tuning, external validation.